# GitHub PR Tracker - Functional Requirements Document

**Project:** GitHub PR Tracker Dashboard  
**Version:** 3.0 (Phase 1 + Phase 2 + Phase 3)  
**Date:** 14 June 2026  
**Status:** Draft (Rev 6)

## 1. Overview

The GitHub PR Tracker is a **team-centric** dashboard application that provides visibility into all **open Pull Requests** raised by members of a configured GitHub team (PHOENIX) **across all repositories** in the organization. Rather than tracking a single repository, the system discovers every repo where a team member has an open PR and presents a unified view. It enables the team to monitor PR status, aging, active reviewer engagement, code owner approval status, and branch targeting at a glance — regardless of which repository the PR belongs to.

## 2. Goals & Objectives

| # | Objective |
|---|---|
| 1 | Provide a single view of all open PRs for the team **across all repositories** in the org |
| 2 | Automatically discover which repositories team members have open PRs in |
| 3 | Highlight how long each PR has been open (aging) with branch-specific thresholds |
| 4 | Show **active** reviewer count per PR (not just assigned/code-owner reviewers) |
| 5 | Identify the PR author and the repository |
| 6 | Enable quick filtering and sorting for prioritization (by repo, branch type, age, etc.) |
| 7 | Distinguish between PRs targeting `main` vs `feature/` branches |
| 8 | Track code owner approval status per PR |
| 9 | Provide a detailed view of each PR without leaving the dashboard |
| 10 | Show team approval progress and unresolved comment counts |

## 3. Inputs / Configuration

| Input | Description | Example | Phase |
|-------|-------------|---------|-------|
| **Organization** | The GitHub organization that owns the team | `nasuni` | 1 |
| ~~**Repository**~~ | ~~Single repo to track~~ | ~~`nasuni/portal`~~ | ~~1 (removed in Phase 3)~~ |
| **Team Slug** | The GitHub team slug to fetch members from | `phoenix` | 1 |
| **GitHub Token** | A PAT or GitHub App token with `repo` + `read:org` scopes | — | 1 |
| **Aging Threshold (Main)** | Days after which a Main PR is considered stale | `3` (default) | 1 |
| **Aging Threshold (Feature)** | Days after which a Feature PR is considered stale | `2` (default) | 1 |
| **Required Team Approvals** | Number of team approvals needed | `2` (default) | 2 |

> **Phase 3 Change:** The `GITHUB_REPO` configuration is **removed**. The system no longer needs a single repository configured — it dynamically discovers all repositories with open PRs from team members using the GitHub Search API.

## 4. Functional Requirements

### FR-1: Fetch Open PRs

| Attribute | Detail |
|-----------|--------|
| **ID** | FR-1 |
| **Title** | Fetch Open Pull Requests |
| **Description** | The system shall fetch all currently open pull requests from the configured GitHub repository. |
| **Acceptance Criteria** | 1. All open PRs are retrieved via the GitHub API. <br> 2. Pagination is handled for repositories with large numbers of PRs. |

---

### FR-2: Resolve Team Members from GitHub Team

| Attribute | Detail |
|-----------|--------|
| **ID** | FR-2 |
| **Title** | Fetch Team Members from GitHub Organization Team (PHOENIX) |
| **Description** | The system shall resolve the list of team members by fetching members of the configured GitHub team (e.g., `PHOENIX`) via the GitHub API (`GET /orgs/{org}/teams/{team_slug}/members`). The fetched PRs are then filtered to only display those opened by these team members. |
| **Acceptance Criteria** | 1. Team members are fetched dynamically from the GitHub team using the API. <br> 2. Only PRs authored by resolved team members appear in the dashboard. <br> 3. If the API call fails (e.g., insufficient permissions), the system falls back to a static/configured list of usernames. <br> 4. The GitHub token must have `read:org` scope for this to work. |
| **Phase 2 Enhancement** | A UI will allow users to search GitHub org members and manually add/remove members from the tracked team list. |

> **Technical Note:** The GitHub API endpoint `GET /orgs/{org}/teams/{team_slug}/members` returns all members of a team. The token requires `read:org` scope. This eliminates the need to manually maintain a user list in Phase 1.

---

### FR-3: Display PR Author

| Attribute | Detail |
|-----------|--------|
| **ID** | FR-3 |
| **Title** | Show PR Author |
| **Description** | For each open PR, the system shall display the GitHub username (and optionally avatar) of the author who opened the PR. |
| **Acceptance Criteria** | 1. Author username is visible for every PR row. <br> 2. Author is correctly mapped to the team member list. |

---

### FR-4: Display PR Age with Branch-Specific Thresholds

| Attribute | Detail |
|-----------|--------|
| **ID** | FR-4 |
| **Title** | Show How Long Each PR Has Been Open (with Branch-Specific Aging Thresholds) |
| **Description** | The system shall calculate and display the duration (in days/hours) since each PR was opened. The aging threshold for visual highlighting differs by branch type: <br> - **Main PRs** (targeting `main`): Stale after **3 days** <br> - **Feature PRs** (targeting `feature/`): Stale after **1-2 days** (default: 2 days) |
| **Acceptance Criteria** | 1. Age is displayed in a human-readable format (e.g., "3d 4h", "5 hours"). <br> 2. Main PRs open longer than **3 days** are visually highlighted (amber/red). <br> 3. Feature PRs open longer than **2 days** are visually highlighted (amber/red). <br> 4. Both thresholds are configurable. <br> 5. Color coding: Normal (green/default) → Warning (amber) → Critical (red) based on how far past the threshold. |

#### Aging Threshold Configuration

| Branch Type | Default Threshold | Rationale |
|-------------|-------------------|----------|
| **Main** (`main`, `master`) | 3 days | Main PRs go through full code owner review and may take longer |
| **Feature** (`feature/*`) | 2 days | Feature PRs are typically smaller, don't require code owner review, and should move faster |

> **Visual Indicator Example:**
> - Main PR open 2 days → Normal (no highlight)
> - Main PR open 4 days → Amber/Warning
> - Feature PR open 1 day → Normal (no highlight)
> - Feature PR open 3 days → Red/Critical

---

### FR-5: Display Active Reviewer Count

| Attribute | Detail |
|-----------|--------|
| **ID** | FR-5 |
| **Title** | Show Number of **Active** Reviewers |
| **Description** | For each PR, the system shall display the count of reviewers who have **actively engaged** with the PR. Active engagement is defined as: submitting a review (approve / request changes), or leaving a comment on the PR. Auto-assigned reviewers (e.g., via CODEOWNERS) who have not taken any action are **excluded** from this count. |
| **Acceptance Criteria** | 1. Only reviewers who have commented, approved, or requested changes are counted. <br> 2. Reviewers who are merely assigned/requested but have not acted are NOT counted. <br> 3. Count is displayed as a numeric value per PR. <br> 4. Data source: PR reviews API (`GET /repos/{owner}/{repo}/pulls/{pull_number}/reviews`) and PR comments. |
| **Rationale** | Code owners are auto-assigned to every PR but do not always actively review. Showing assigned count would be misleading; active count reflects actual review engagement. |

---

### FR-6: Display PR Metadata

| Attribute | Detail |
|-----------|--------|
| **ID** | FR-6 |
| **Title** | Show PR Title, Number, and Link |
| **Description** | Each PR in the dashboard shall display its title, PR number, and a clickable link to the PR on GitHub. |
| **Acceptance Criteria** | 1. PR title is shown. <br> 2. PR number (e.g., #142) is shown. <br> 3. Clicking the PR opens it in a new browser tab on GitHub. |

---

### FR-7: Sort PRs

| Attribute | Detail |
|-----------|--------|
| **ID** | FR-7 |
| **Title** | Sort PRs by Age, Author, or Active Reviewer Count |
| **Description** | The dashboard shall allow sorting of the PR list by age (oldest first), author name, or active reviewer count. |
| **Acceptance Criteria** | 1. Default sort is by age (oldest first). <br> 2. User can toggle between sort criteria. |

---

### FR-8: Auto-Refresh / Manual Refresh

| Attribute | Detail |
|-----------|--------|
| **ID** | FR-8 |
| **Title** | Refresh PR Data |
| **Description** | The dashboard shall support refreshing the data either on a configurable interval (auto-refresh) or via a manual refresh button. |
| **Acceptance Criteria** | 1. A refresh button is available. <br> 2. Optionally, data refreshes automatically every N minutes (configurable). |

---

### FR-9: Filter by Base Branch (Main vs Feature)

| Attribute | Detail |
|-----------|--------|
| **ID** | FR-9 |
| **Title** | Filter PRs by Target Base Branch |
| **Description** | The dashboard shall categorize and allow filtering of PRs based on their target (base) branch. PRs are classified as: <br> - **Main PRs** — base branch is `main` (or `master`) <br> - **Feature PRs** — base branch starts with `feature/` |
| **Acceptance Criteria** | 1. Each PR displays a branch type indicator (badge/chip: "Main" or "Feature"). <br> 2. The dashboard provides tab/toggle filters: **All** \| **Main** \| **Feature Branches**. <br> 3. Classification rule: if base branch starts with `feature/`, it is a Feature PR; otherwise it is a Main PR. <br> 4. The selected filter persists during the session. |
| **Rationale** | Feature branch PRs do not require code owners review and have different review dynamics. Separating them helps the team focus on the PRs relevant to their workflow. |
| **UX Recommendation** | Use a **single table with tab filters** (All / Main / Feature) rather than two separate tables. This keeps the UI clean, avoids layout duplication, and lets users see everything in one place or drill down as needed. |

---

### FR-10: Code Owner Approval Status

| Attribute | Detail |
|-----------|--------|
| **ID** | FR-10 |
| **Title** | Track and Display Code Owner Approval Status |
| **Description** | For each PR (targeting `main`), the system shall determine whether at least one code owner has approved the PR. The system dynamically identifies code owners by fetching and parsing the repository's `CODEOWNERS` file, then cross-referencing with the PR's reviews. |
| **Acceptance Criteria** | 1. The system fetches the `CODEOWNERS` file from the repository (checks `.github/CODEOWNERS`, `CODEOWNERS`, and `docs/CODEOWNERS`). <br> 2. The CODEOWNERS file is parsed to extract ownership rules (glob patterns → users/teams). <br> 3. For each PR, the system fetches the list of changed files (`GET /repos/{owner}/{repo}/pulls/{pull_number}/files`). <br> 4. The system matches changed files against CODEOWNERS patterns to identify required code owners. <br> 5. The system checks PR reviews (`GET /repos/{owner}/{repo}/pulls/{pull_number}/reviews`) to determine if any identified code owner has approved. <br> 6. A status indicator is displayed: ✅ (approved by code owner) or ⏳ (pending code owner approval). |

#### How Code Owners Are Identified Dynamically

```
Step 1: Fetch CODEOWNERS file
  → GET /repos/{owner}/{repo}/contents/.github/CODEOWNERS
  → Fallback: /CODEOWNERS or /docs/CODEOWNERS

Step 2: Parse CODEOWNERS rules
  → Each line: <glob-pattern> <@owner1> <@owner2> <@org/team>
  → Example: "*.js @frontend-team @user1"

Step 3: Fetch changed files in PR
  → GET /repos/{owner}/{repo}/pulls/{pull_number}/files
  → Returns list of modified file paths

Step 4: Match files → owners
  → For each changed file, find matching CODEOWNERS patterns
  → Collect the set of required code owners (users + teams)
  → If a team is listed (e.g., @org/team-name), resolve team members via:
     GET /orgs/{org}/teams/{team_slug}/members

Step 5: Check reviews for code owner approval
  → GET /repos/{owner}/{repo}/pulls/{pull_number}/reviews
  → If any reviewer in the code owners set has state = "APPROVED" → ✅
  → Otherwise → ⏳
```

> **Note:** There is no single GitHub API that directly tells you "code owner has approved." We must compose this from the CODEOWNERS file + changed files + reviews. The CODEOWNERS file is fetched once and cached (it changes infrequently). The file parsing follows the same glob-matching rules GitHub uses internally.

---

### FR-11: PR Detail Panel (Side Drawer)

| Attribute | Detail |
|-----------|--------|
| **ID** | FR-11 |
| **Title** | Show Detailed PR View in Side Panel |
| **Description** | When a user clicks on a PR row in the table, a **side panel (drawer)** shall slide in from the right, displaying detailed information about that PR without navigating away from the dashboard. |
| **Acceptance Criteria** | 1. Clicking a PR row opens a right-side drawer panel. <br> 2. The main table remains visible (narrowed) for context. <br> 3. The drawer can be dismissed by clicking outside, pressing Escape, or a close button. <br> 4. The drawer displays the detailed information listed below. |

#### Detail Panel Contents

| # | Section | Data Shown |
|---|---------|------------|
| 1 | **Header** | PR title, number, author avatar + username, branch badge (Main/Feature) |
| 2 | **Status Bar** | Age (with color indicator), code owner approval status (✅/⏳) |
| 3 | **Description** | PR body/description (rendered Markdown, truncated with "show more") |
| 4 | **Branches** | Source branch → Base branch |
| 5 | **Reviewers** | List of active reviewers with their review status (approved ✅ / changes requested 🔄 / commented 💬) |
| 6 | **Code Owners** | List of identified code owners for this PR and whether they've approved |
| 7 | **Files Changed** | Count of files changed (additions/deletions summary) |
| 8 | **Labels** | Any labels applied to the PR |
| 9 | **Actions** | "Open in GitHub" button (new tab) |

#### UX Design Rationale

| Pattern | Pros | Cons | **Decision** |
|---------|------|------|--------------|
| Expandable Row (accordion) | Simple, inline | Limited space, pushes other rows down | ❌ |
| Side Panel (drawer) | Table stays visible, ample detail space, standard pattern | Slightly more complex to implement | ✅ **Selected** |
| Modal / Dialog | Focused view | Blocks table, feels disruptive | ❌ |
| Separate Page | Maximum space | Loses table context, extra navigation | ❌ |

> **Recommendation:** The **right-side drawer** is the best pattern for this use case. It allows users to quickly scan PR details while keeping the full table visible for context. This is a standard dashboard pattern (used by Jira, Linear, GitHub Projects, etc.).

## 5. Dashboard View - Data Columns

The dashboard uses a **single table with tab filters** (All | Main | Feature Branches) and a **repository filter** (Phase 3).

| # | Column | Description | Phase |
|---|--------|-------------|-------|
| 1 | **PR Number** | The pull request number (e.g., #142) | 1 |
| 2 | **Repository** | Short repo name (e.g., `portal`, `nmc-api`) | **3** |
| 3 | **Title** | The title of the pull request (with unresolved comment indicator if > 0) | 1 (indicator: Phase 2) |
| 4 | **Author** | GitHub username of the PR creator | 1 |
| 5 | **Base Branch** | Target branch with type badge (Main / Feature) | 1 |
| 6 | **Age** | Time since PR was opened (e.g., "3d 4h"), color-coded per branch-specific threshold | 1 |
| 7 | **Active Reviewers** | Count of reviewers who have actively engaged | 1 |
| 8 | **Team Approvals** | `N/M` showing team approvals vs required, with sufficiency indicator | 2 |
| 9 | **Code Owner Status** | ✅ Approved / ⏳ Pending (shown for Main PRs) | 1 |
| 10 | **Link** | Direct URL to the PR on GitHub | 1 |

**Filter Tabs:** All | Main | Feature Branches (client-side filtering)  
**Repository Filter:** Dropdown/multi-select to filter by repo (Phase 3, client-side)  
**Row Click:** Opens the PR Detail Panel (side drawer)  
**Sorting:** Client-side instant sort

## 6. User Stories

### US-1: View Open PRs
> **As a** team member,  
> **I want to** see all open PRs from my team in one dashboard,  
> **So that** I can quickly identify which PRs need attention.

### US-2: Identify Stale PRs
> **As a** team lead,  
> **I want to** see how long each PR has been open, with different staleness thresholds for main vs feature PRs,  
> **So that** I can follow up on PRs that are aging relative to their expected lifecycle.

### US-3: Check Active Reviewer Engagement
> **As a** PR author,  
> **I want to** see how many reviewers have **actively** reviewed my PR (commented/approved/requested changes),  
> **So that** I know if my PR is actually being reviewed or if I need to follow up.

### US-4: Quick Navigation
> **As a** team member,  
> **I want to** click on a PR and be taken directly to GitHub,  
> **So that** I can review or take action on it immediately.

### US-5: Filter by Branch Type
> **As a** team member,  
> **I want to** filter PRs by whether they target `main` or a `feature/` branch,  
> **So that** I can focus on the PRs relevant to my current review workflow.

### US-6: Automatic Team Resolution
> **As a** team lead,  
> **I want** the dashboard to automatically fetch team members from our GitHub team (PHOENIX),  
> **So that** I don't have to manually maintain a list of usernames.

### US-7: Check Code Owner Approval
> **As a** PR author,  
> **I want to** see whether a code owner has approved my PR,  
> **So that** I know if the PR is ready to merge from a code ownership perspective.

### US-8: View PR Details
> **As a** team member,  
> **I want to** click on a PR row and see detailed information in a side panel,  
> **So that** I can get full context (description, reviewers, files changed) without leaving the dashboard.

### US-9: Instant Filtering (Phase 2)
> **As a** team member,  
> **I want** tab switching and sorting to be instant (no loading spinners),  
> **So that** I can quickly scan different views without waiting for network requests.

### US-10: Check Team Approval Readiness (Phase 2)
> **As a** PR author,  
> **I want to** see at a glance whether my PR has enough team approvals to ping the code owner,  
> **So that** I know when to escalate for final review.

### US-11: Identify PRs with Unresolved Feedback (Phase 2)
> **As a** team member,  
> **I want to** see which PRs have unresolved review comments,  
> **So that** I know which PRs are blocked on author response and which are ready for additional review.

## 7. Phase 2 Requirements

Phase 2 adds team approval visibility, unresolved comment indicators, and moves filtering/sorting to the client side.

---

### FR-12: Client-Side Filtering & Sorting

| Attribute | Detail |
|-----------|--------|
| **ID** | FR-12 |
| **Title** | Move Filtering and Sorting to Client Side |
| **Description** | Currently, every filter/tab change (All / Main / Feature) and sort toggle triggers a new backend API call. Since the dataset is small (team's open PRs, typically < 30) and auto-refresh already fetches all data every 2 minutes, filtering and sorting shall be performed client-side on the already-fetched data. The backend **retains** the `branch_type`, `sort_by`, `sort_order` query parameters for backward compatibility and other API consumers, but the frontend no longer sends them — it fetches all PRs once and filters/sorts locally. |
| **Acceptance Criteria** | 1. Backend `/api/v1/pull-requests` endpoint **still supports** `branch_type`, `sort_by`, `sort_order` query parameters (unchanged API contract). <br> 2. Frontend calls the endpoint **without** any filter/sort parameters — always fetches all open team PRs. <br> 3. Frontend filters and sorts the cached dataset client-side (via `useMemo` or equivalent). <br> 4. Switching tabs (All / Main / Feature) is **instant** — no loading spinner, no network request. <br> 5. Sorting by clicking column headers is **instant** — no network request. <br> 6. Auto-refresh continues to fetch all data every 2 minutes in the background. <br> 7. Tab badge counts are computed from the full dataset client-side. |
| **Rationale** | Eliminates unnecessary API calls on every user interaction. The dataset is small enough to hold in memory. Provides a snappier UX with zero latency on filter/sort changes. Backend params are retained for API consumers (scripts, other tools). |

---

### FR-13: Team Approvals Column

| Attribute | Detail |
|-----------|--------|
| **ID** | FR-13 |
| **Title** | Display Team Approval Count with Reviewer Names on Hover |
| **Description** | For each PR, the dashboard shall display the number of PHOENIX team member approvals received vs. the number required. **Hovering** on the count shows the names of team members who approved. This helps the team know when a PR has enough peer approvals to warrant pinging the code owner for final review. |
| **Acceptance Criteria** | 1. A "Team Approvals" column is displayed in the PR table, positioned between "Reviewers" and "Code Owner." <br> 2. The column shows `N / M` where N = approvals received from PHOENIX team members, M = required count (configurable, default: 2). <br> 3. When N ≥ M: display a green checkmark/badge indicating "sufficient approvals — ready for code owner ping." <br> 4. When N < M: display the count in neutral/yellow styling. <br> 5. **Hovering** on the approval count shows a tooltip with the **names** of team members who have approved. <br> 6. Only `APPROVED` reviews from **PHOENIX team members** are counted (not the PR author, not code owners). <br> 7. Team membership is determined by the configured GitHub team (PHOENIX) — the same team used to filter PRs. <br> 8. The required approval count is configurable via `REQUIRED_TEAM_APPROVALS` env var (default: `2`). |
| **Rationale** | Allows the team to quickly identify which PRs are ready for code owner escalation. The tooltip with names gives visibility into who has already reviewed without opening the PR. |

#### Team Approval Rules

| Rule | Detail |
|------|--------|
| Who counts | Members of the PHOENIX GitHub team who submitted `APPROVED` review |
| Who is excluded | The PR author (even if they're a team member), code owners |
| How team is identified | Same PHOENIX team used for PR filtering (fetched via `GET /orgs/{org}/teams/{team_slug}/members`) |
| Required count | Configurable via `REQUIRED_TEAM_APPROVALS` (default: 2) |
| Visual: sufficient | Green badge: `2/2 ✓` |
| Visual: insufficient | Neutral/yellow: `1/2` |
| Hover/tooltip | Shows names of approving team members (e.g., "Approved by: alice, bob") |

---

### FR-14: Unresolved Comments Indicator

| Attribute | Detail |
|-----------|--------|
| **ID** | FR-14 |
| **Title** | Display Unresolved Review Comment Count |
| **Description** | For each PR, the dashboard shall indicate whether there are unresolved review threads. This signals that the PR author has pending feedback to address before the PR can be merged. |
| **Acceptance Criteria** | 1. If a PR has 0 unresolved comments, nothing is shown (clean row). <br> 2. If a PR has 1+ unresolved comments, a comment bubble icon + count is displayed (e.g., `💬 3`). <br> 3. The count reflects unresolved **review threads** (not individual comments within a thread). <br> 4. Styling: orange/amber for 1-2 unresolved, red for 3+. <br> 5. The indicator is shown inline with the PR title or as a compact dedicated column. |
| **Rationale** | PRs with unresolved comments need author attention before further reviews are productive. This gives the team visibility into which PRs are blocked on author response. |

#### Unresolved Comment Display Rules

| State | Display | Style |
|-------|---------|-------|
| 0 unresolved | Nothing (clean) | — |
| 1-2 unresolved | 💬 N | Orange/amber |
| 3+ unresolved | 💬 N | Red |

---

### FR-15: Updated Detail Panel (Phase 2 Additions)

| Attribute | Detail |
|-----------|--------|
| **ID** | FR-15 |
| **Title** | Add Team Approvals & Unresolved Comments to PR Detail Panel |
| **Description** | The PR detail side drawer (FR-11) shall be enhanced to show the full team approvals breakdown and the list of unresolved comment threads. |
| **Acceptance Criteria** | 1. The detail panel shows a **"Team Approvals"** section listing each PHOENIX team member who approved (with ✅) and those who haven't reviewed yet (with —). <br> 2. Shows the `N/M` sufficiency indicator with a clear "Ready for code owner review" or "Needs more team reviews" message. <br> 3. The detail panel shows an **"Unresolved Comments"** section listing each unresolved thread with: commenter name, file path, and a snippet of the comment text. <br> 4. Each unresolved thread is clickable/linked to the specific comment on GitHub. <br> 5. If there are no unresolved comments, this section shows "No unresolved comments ✓". |

#### Updated Detail Panel Contents (Phase 2)

| # | Section | Data Shown | Phase |
|---|---------|------------|-------|
| 1 | **Header** | PR title, number, author avatar + username, branch badge (Main/Feature) | 1 |
| 2 | **Status Bar** | Age (with color indicator), code owner approval status (✅/⏳), unresolved count badge | 1 + **2** |
| 3 | **Description** | PR body/description (rendered Markdown, truncated with "show more") | 1 |
| 4 | **Branches** | Source branch → Base branch | 1 |
| 5 | **Team Approvals** | List of PHOENIX team members with approval status; `N/M` indicator; "Ready for CO" message | **2** |
| 6 | **Reviewers** | List of active reviewers with their review status (approved ✅ / changes requested 🔄 / commented 💬) | 1 |
| 7 | **Unresolved Comments** | List of unresolved threads: commenter, file, snippet, link to GitHub | **2** |
| 8 | **Code Owners** | List of identified code owners for this PR and whether they've approved | 1 |
| 9 | **Files Changed** | Count of files changed (additions/deletions summary) | 1 |
| 10 | **Labels** | Any labels applied to the PR | 1 |
| 11 | **Actions** | "Open in GitHub" button (new tab) | 1 |

## 10. Phase 3 Requirements — Multi-Repository Team Tracking

Phase 3 transforms the tracker from a single-repo tool into a **team-wide PR tracker** that discovers and displays PRs across all repositories where PHOENIX team members have open PRs.

---

### FR-16: Multi-Repository PR Discovery

| Attribute | Detail |
|-----------|--------|
| **ID** | FR-16 |
| **Title** | Discover and Fetch PRs Across All Repositories |
| **Description** | The system shall use the GitHub Search API to find all open pull requests authored by any PHOENIX team member across all repositories in the configured organization. The single `GITHUB_REPO` config is removed — the system dynamically discovers repos. |
| **Acceptance Criteria** | 1. The system searches for open PRs by each team member across the entire org using: `GET /search/issues?q=is:pr+is:open+org:{org}+author:{username}`. <br> 2. Results are aggregated across all team members (deduplicated by PR number + repo). <br> 3. No hardcoded repository list — repos are auto-discovered from search results. <br> 4. Pagination is handled (Search API returns max 100 per page). <br> 5. All enrichment (age, reviewers, code owners, team approvals, unresolved threads) works per-PR regardless of repository. <br> 6. The `GITHUB_REPO` environment variable is **no longer required** (backward-compatible: if set, acts as a filter to only show that repo). |
| **API Strategy** | Use `GET /search/issues` with query `is:pr is:open org:{org} author:{user1} OR author:{user2} ...` to fetch all team PRs in a single paginated query. This is more efficient than per-repo fetching. |

#### Technical Approach: GitHub Search API

```
Strategy A (Preferred): Single search query with OR'd authors
  → GET /search/issues?q=is:pr+is:open+org:nasuni+author:user1+OR+author:user2+OR+author:user3...
  → Returns PRs from ALL repos in one paginated response
  → Rate limit: 30 requests/minute for search (authenticated)
  → Max 1000 results (sufficient for team open PRs)

Strategy B (Fallback): Per-member queries
  → One query per team member: is:pr+is:open+org:nasuni+author:{username}
  → Aggregate results
  → Higher API call count but works if query length exceeds limits

Enrichment (per PR):
  → Reviews: GET /repos/{owner}/{repo}/pulls/{number}/reviews
  → Files: GET /repos/{owner}/{repo}/pulls/{number}/files
  → Threads: GraphQL reviewThreads query (same as Phase 2)
  → CODEOWNERS: GET /repos/{owner}/{repo}/contents/.github/CODEOWNERS (cached per repo)
```

---

### FR-17: Repository Column & Grouping

| Attribute | Detail |
|-----------|--------|
| **ID** | FR-17 |
| **Title** | Display Repository Name for Each PR |
| **Description** | Since PRs now span multiple repositories, each PR row shall display the repository it belongs to. The dashboard shall also support filtering by repository. |
| **Acceptance Criteria** | 1. A **"Repository"** column is displayed in the PR table (short name, e.g., `portal`, `nmc-api`, not full path). <br> 2. The column is positioned after PR Number and before Title. <br> 3. Clicking the repo name could filter to show only that repo's PRs (or link to the repo on GitHub). <br> 4. A **repository filter** (dropdown or multi-select) is available alongside the branch type tabs. <br> 5. Filter options are dynamically populated from the repos found in the current dataset. <br> 6. The "All" tab shows PRs across all repos; selecting a specific repo filters to that repo only. <br> 7. Client-side filtering applies to repo filter (same pattern as branch type tabs). |

#### Updated Dashboard Columns (Phase 3)

| # | Column | Description | Phase |
|---|--------|-------------|-------|
| 1 | **PR Number** | e.g., #142 | 1 |
| 2 | **Repository** | Short repo name (e.g., `portal`) | **3** |
| 3 | **Title** | PR title (with unresolved comment indicator) | 1 |
| 4 | **Author** | GitHub username | 1 |
| 5 | **Base Branch** | Target branch with badge (Main / Feature) | 1 |
| 6 | **Age** | Time since opened, color-coded | 1 |
| 7 | **Active Reviewers** | Count of actively engaged reviewers | 1 |
| 8 | **Team Approvals** | `N/M` with sufficiency indicator | 2 |
| 9 | **Code Owner Status** | ✅ Approved / ⏳ Pending (Main PRs only) | 1 |
| 10 | **Link** | Direct URL to GitHub | 1 |

---

### FR-18: Repository Filter UI

| Attribute | Detail |
|-----------|--------|
| **ID** | FR-18 |
| **Title** | Filter PRs by Repository |
| **Description** | The dashboard shall provide a repository filter that allows users to view PRs from all repos or narrow down to specific repositories. |
| **Acceptance Criteria** | 1. A repository filter control is displayed above or beside the branch type tabs. <br> 2. Default selection is "All Repositories." <br> 3. The filter lists all repos that have at least one open team PR (dynamically populated). <br> 4. Selecting a repo instantly filters the table (client-side). <br> 5. The repo filter works in combination with branch type tabs (both can be active simultaneously). <br> 6. Badge counts on branch tabs update to reflect the currently selected repo filter. <br> 7. The filter shows the count of PRs per repo (e.g., `portal (7)`, `nmc-api (3)`). |

---

### FR-19: Per-Repository Code Owner Handling

| Attribute | Detail |
|-----------|--------|
| **ID** | FR-19 |
| **Title** | Fetch and Cache CODEOWNERS Per Repository |
| **Description** | Since each repository may have its own CODEOWNERS file, the system shall fetch and cache CODEOWNERS content independently for each repository encountered. |
| **Acceptance Criteria** | 1. CODEOWNERS file is fetched per repository (not globally). <br> 2. Results are cached for the duration of the refresh cycle (2 minutes). <br> 3. If a repository has no CODEOWNERS file, code owner status shows "—" for that repo's PRs. <br> 4. The same parsing logic (glob matching, owner resolution) applies regardless of repo. |

---

### FR-20: Removal of Single-Repo Configuration

| Attribute | Detail |
|-----------|--------|
| **ID** | FR-20 |
| **Title** | Deprecate and Remove `GITHUB_REPO` Requirement |
| **Description** | The `GITHUB_REPO` environment variable shall no longer be required. The system discovers repositories dynamically. For backward compatibility, if `GITHUB_REPO` is set, it acts as an **optional filter** to restrict tracking to that single repo. |
| **Acceptance Criteria** | 1. The system starts and operates correctly **without** `GITHUB_REPO` set. <br> 2. If `GITHUB_REPO` is set, only PRs from that repo are shown (legacy single-repo mode). <br> 3. If `GITHUB_REPO` is not set (or empty), PRs from all org repos are shown. <br> 4. The dashboard header/title updates to reflect scope: "PHOENIX Team PRs" (multi-repo) vs "PHOENIX PRs — portal" (single-repo). |

---

### FR-21: Search API Rate Limit Handling

| Attribute | Detail |
|-----------|--------|
| **ID** | FR-21 |
| **Title** | Handle GitHub Search API Rate Limits |
| **Description** | The GitHub Search API has a stricter rate limit (30 requests/minute for authenticated users) compared to the REST API (5000/hour). The system shall handle this gracefully. |
| **Acceptance Criteria** | 1. The system respects the 30 req/min search rate limit. <br> 2. If rate-limited, the system uses cached data from the previous successful fetch. <br> 3. Rate limit status (remaining/reset) is included in the health endpoint. <br> 4. The auto-refresh interval (2 min) is sufficient to stay within limits for typical team sizes. <br> 5. Error states are surfaced to the UI (e.g., "Data may be stale — rate limit reached"). |

## 11. Phase 3 — User Stories

### US-12: View Team PRs Across All Repos
> **As a** team member,  
> **I want to** see all open PRs from my team across every repository in the org,  
> **So that** I have a single place to track all team activity regardless of which repo it's in.

### US-13: Identify Which Repo a PR Belongs To
> **As a** team member,  
> **I want to** see which repository each PR belongs to,  
> **So that** I can quickly understand the context and scope of the change.

### US-14: Filter by Repository
> **As a** team member,  
> **I want to** filter the dashboard to show PRs from a specific repository,  
> **So that** I can focus on the repos I'm responsible for reviewing.

### US-15: Zero-Config Repo Discovery
> **As a** team lead,  
> **I want** the system to automatically discover repos with team PRs (no configuration needed),  
> **So that** when the team starts working on a new repo, it automatically appears in the tracker.

### US-16: Backward Compatibility
> **As an** admin,  
> **I want** the option to restrict tracking to a single repo if needed,  
> **So that** existing single-repo deployments continue to work without changes.

## 12. Phase 3 — Updated Assumptions

1. All previous assumptions (Phase 1 + 2) remain valid.
2. The GitHub token has `repo` scope which grants access to read PRs, reviews, and files across **all** repos in the org (public and private).
3. The GitHub Search API (authenticated) allows 30 requests/minute — sufficient for a 2-minute refresh cycle.
4. The team's total open PRs across all repos is expected to be under 100 (well within the 1000-result search limit).
5. CODEOWNERS files differ per repository and must be fetched/parsed independently.
6. The GitHub Search API returns PRs from all repos the token can access (private repos included if token has `repo` scope).
7. Repository names are unique within the org (standard GitHub constraint).

---

## 13. Phase 3 — Impact Summary

| Area | Change |
|------|--------|
| **Configuration** | `GITHUB_REPO` becomes optional (backward-compat filter). No new required config. |
| **Data Source** | Switch from `GET /repos/{owner}/{repo}/pulls` to `GET /search/issues` with org-wide query. |
| **Enrichment** | Same per-PR enrichment (reviews, files, threads, CODEOWNERS) — now parameterized by repo. |
| **Frontend — Table** | New "Repository" column between PR# and Title. |
| **Frontend — Filters** | New repo filter dropdown (multi-select or single-select) alongside branch tabs. |
| **Frontend — Header** | Title updates to "PHOENIX Team PRs" (multi-repo) or "PHOENIX PRs — {repo}" (single-repo). |
| **Caching** | CODEOWNERS cached per-repo. Search results cached between refreshes. |
| **Rate Limits** | Search API: 30/min (monitor via health endpoint). REST API: 5000/hr (unchanged). |

## 8. Assumptions

1. A GitHub team named **PHOENIX** (slug: `phoenix`) exists in the organization and contains all relevant team members.
2. A valid GitHub token with `repo` + `read:org` scopes will be provided.
3. The target repository is hosted on GitHub.com (not GitHub Enterprise, unless token supports it).
4. The dashboard will be accessed by team members within a browser.
5. PRs targeting branches starting with `feature/` are considered feature branch PRs; all others are considered main PRs.
6. Code owners are auto-assigned as reviewers but should NOT count toward active review engagement unless they take action.
7. A `CODEOWNERS` file exists in the repository (in `.github/`, root, or `docs/` directory).
8. The CODEOWNERS file follows standard GitHub CODEOWNERS syntax (glob patterns).
9. Feature branch PRs have a tighter aging expectation (1-2 days) than main branch PRs (3 days) because they are typically smaller and don't require code owner review.

## 9. Open Questions

| # | Question | Status |
|---|----------|--------|
| 1 | Which GitHub organization and repository (org/repo) should be tracked? | **Pending - input from user** |
| 2 | Confirm team slug is `phoenix`? | **Pending - input from user** |
| 3 | Is this GitHub.com or GitHub Enterprise? | Pending |
| 4 | Any authentication preferences (PAT vs GitHub App)? | Pending |
| 5 | Are there base branches other than `main` and `feature/*` that need special handling? | Pending |
| 6 | Feature branch aging: prefer 1 day or 2 days as default threshold? | Default: 2 days |

---

## Next Steps

1. **User to confirm:** Organization, repository, and team slug.
2. **Next document:** Technical Requirements Document (architecture, tech stack, API design).
3. **Then:** Implementation.

# GitHub PR Tracker - Functional Requirements Document

**Project:** GitHub PR Tracker Dashboard  
**Version:** 1.0 (Phase 1)  
**Date:** 2 June 2026  
**Status:** Draft (Rev 3)

## 1. Overview

The GitHub PR Tracker is a dashboard application that provides visibility into all **open Pull Requests** for a given GitHub repository, filtered to PRs opened by members of a configured GitHub team (PHOENIX). It enables the team to monitor PR status, aging, active reviewer engagement, code owner approval status, and branch targeting at a glance.

## 2. Goals & Objectives

| # | Objective |
|---|---|
| 1 | Provide a single view of all open PRs for the team across a repository |
| 2 | Highlight how long each PR has been open (aging) |
| 3 | Show **active** reviewer count per PR (not just assigned/code-owner reviewers) |
| 4 | Identify the PR author |
| 5 | Enable quick filtering and sorting for prioritization |
| 6 | Distinguish between PRs targeting `main` vs `feature/` branches |
| 7 | Track code owner approval status per PR |
| 8 | Provide a detailed view of each PR without leaving the dashboard |

## 3. Inputs / Configuration

| Input | Description | Example |
|-------|-------------|--------|
| **Organization** | The GitHub organization that owns the repo and team | `my-org` |
| **Repository** | The GitHub repository to track (owner/repo) | `my-org/my-repo` |
| **Team Slug** | The GitHub team slug to fetch members from | `phoenix` |
| **GitHub Token** | A PAT or GitHub App token with `repo` + `read:org` scopes | — |
| **Aging Threshold** | Number of days after which a PR is considered stale (default: 3) | `3` |

## 4. Functional Requirements

### FR-1: Fetch Open PRs

| Attribute | Detail |
|-----------|--------|
| **ID** | FR-1 |
| **Title** | Fetch Open Pull Requests |
| **Description** | The system shall fetch all currently open pull requests from the configured GitHub repository. |
| **Acceptance Criteria** | 1. All open PRs are retrieved via the GitHub API. <br> 2. Pagination is handled for repositories with large numbers of PRs. |

---

### FR-2: Resolve Team Members from GitHub Team

| Attribute | Detail |
|-----------|--------|
| **ID** | FR-2 |
| **Title** | Fetch Team Members from GitHub Organization Team (PHOENIX) |
| **Description** | The system shall resolve the list of team members by fetching members of the configured GitHub team (e.g., `PHOENIX`) via the GitHub API (`GET /orgs/{org}/teams/{team_slug}/members`). The fetched PRs are then filtered to only display those opened by these team members. |
| **Acceptance Criteria** | 1. Team members are fetched dynamically from the GitHub team using the API. <br> 2. Only PRs authored by resolved team members appear in the dashboard. <br> 3. If the API call fails (e.g., insufficient permissions), the system falls back to a static/configured list of usernames. <br> 4. The GitHub token must have `read:org` scope for this to work. |
| **Phase 2 Enhancement** | A UI will allow users to search GitHub org members and manually add/remove members from the tracked team list. |

> **Technical Note:** The GitHub API endpoint `GET /orgs/{org}/teams/{team_slug}/members` returns all members of a team. The token requires `read:org` scope. This eliminates the need to manually maintain a user list in Phase 1.

---

### FR-3: Display PR Author

| Attribute | Detail |
|-----------|--------|
| **ID** | FR-3 |
| **Title** | Show PR Author |
| **Description** | For each open PR, the system shall display the GitHub username (and optionally avatar) of the author who opened the PR. |
| **Acceptance Criteria** | 1. Author username is visible for every PR row. <br> 2. Author is correctly mapped to the team member list. |

---

### FR-4: Display PR Age (Time Open)

| Attribute | Detail |
|-----------|--------|
| **ID** | FR-4 |
| **Title** | Show How Long Each PR Has Been Open |
| **Description** | The system shall calculate and display the duration (in days/hours) since each PR was opened. |
| **Acceptance Criteria** | 1. Age is displayed in a human-readable format (e.g., "3 days", "5 hours"). <br> 2. PRs open longer than a configurable threshold (default: 3 days) are visually highlighted (e.g., red/amber). |

---

### FR-5: Display Active Reviewer Count

| Attribute | Detail |
|-----------|--------|
| **ID** | FR-5 |
| **Title** | Show Number of **Active** Reviewers |
| **Description** | For each PR, the system shall display the count of reviewers who have **actively engaged** with the PR. Active engagement is defined as: submitting a review (approve / request changes), or leaving a comment on the PR. Auto-assigned reviewers (e.g., via CODEOWNERS) who have not taken any action are **excluded** from this count. |
| **Acceptance Criteria** | 1. Only reviewers who have commented, approved, or requested changes are counted. <br> 2. Reviewers who are merely assigned/requested but have not acted are NOT counted. <br> 3. Count is displayed as a numeric value per PR. <br> 4. Data source: PR reviews API (`GET /repos/{owner}/{repo}/pulls/{pull_number}/reviews`) and PR comments. |
| **Rationale** | Code owners are auto-assigned to every PR but do not always actively review. Showing assigned count would be misleading; active count reflects actual review engagement. |

---

### FR-6: Display PR Metadata

| Attribute | Detail |
|-----------|--------|
| **ID** | FR-6 |
| **Title** | Show PR Title, Number, and Link |
| **Description** | Each PR in the dashboard shall display its title, PR number, and a clickable link to the PR on GitHub. |
| **Acceptance Criteria** | 1. PR title is shown. <br> 2. PR number (e.g., #142) is shown. <br> 3. Clicking the PR opens it in a new browser tab on GitHub. |

---

### FR-7: Sort PRs

| Attribute | Detail |
|-----------|--------|
| **ID** | FR-7 |
| **Title** | Sort PRs by Age, Author, or Active Reviewer Count |
| **Description** | The dashboard shall allow sorting of the PR list by age (oldest first), author name, or active reviewer count. |
| **Acceptance Criteria** | 1. Default sort is by age (oldest first). <br> 2. User can toggle between sort criteria. |

---

### FR-8: Auto-Refresh / Manual Refresh

| Attribute | Detail |
|-----------|--------|
| **ID** | FR-8 |
| **Title** | Refresh PR Data |
| **Description** | The dashboard shall support refreshing the data either on a configurable interval (auto-refresh) or via a manual refresh button. |
| **Acceptance Criteria** | 1. A refresh button is available. <br> 2. Optionally, data refreshes automatically every N minutes (configurable). |

---

### FR-9: Filter by Base Branch (Main vs Feature)

| Attribute | Detail |
|-----------|--------|
| **ID** | FR-9 |
| **Title** | Filter PRs by Target Base Branch |
| **Description** | The dashboard shall categorize and allow filtering of PRs based on their target (base) branch. PRs are classified as: <br> - **Main PRs** — base branch is `main` (or `master`) <br> - **Feature PRs** — base branch starts with `feature/` |
| **Acceptance Criteria** | 1. Each PR displays a branch type indicator (badge/chip: "Main" or "Feature"). <br> 2. The dashboard provides tab/toggle filters: **All** \| **Main** \| **Feature Branches**. <br> 3. Classification rule: if base branch starts with `feature/`, it is a Feature PR; otherwise it is a Main PR. <br> 4. The selected filter persists during the session. |
| **Rationale** | Feature branch PRs do not require code owners review and have different review dynamics. Separating them helps the team focus on the PRs relevant to their workflow. |
| **UX Recommendation** | Use a **single table with tab filters** (All / Main / Feature) rather than two separate tables. This keeps the UI clean, avoids layout duplication, and lets users see everything in one place or drill down as needed. |

---

### FR-10: Code Owner Approval Status

| Attribute | Detail |
|-----------|--------|
| **ID** | FR-10 |
| **Title** | Track and Display Code Owner Approval Status |
| **Description** | For each PR (targeting `main`), the system shall determine whether at least one code owner has approved the PR. The system dynamically identifies code owners by fetching and parsing the repository's `CODEOWNERS` file, then cross-referencing with the PR's reviews. |
| **Acceptance Criteria** | 1. The system fetches the `CODEOWNERS` file from the repository (checks `.github/CODEOWNERS`, `CODEOWNERS`, and `docs/CODEOWNERS`). <br> 2. The CODEOWNERS file is parsed to extract ownership rules (glob patterns → users/teams). <br> 3. For each PR, the system fetches the list of changed files (`GET /repos/{owner}/{repo}/pulls/{pull_number}/files`). <br> 4. The system matches changed files against CODEOWNERS patterns to identify required code owners. <br> 5. The system checks PR reviews (`GET /repos/{owner}/{repo}/pulls/{pull_number}/reviews`) to determine if any identified code owner has approved. <br> 6. A status indicator is displayed: ✅ (approved by code owner) or ⏳ (pending code owner approval). |

#### How Code Owners Are Identified Dynamically

```
Step 1: Fetch CODEOWNERS file
  → GET /repos/{owner}/{repo}/contents/.github/CODEOWNERS
  → Fallback: /CODEOWNERS or /docs/CODEOWNERS

Step 2: Parse CODEOWNERS rules
  → Each line: <glob-pattern> <@owner1> <@owner2> <@org/team>
  → Example: "*.js @frontend-team @user1"

Step 3: Fetch changed files in PR
  → GET /repos/{owner}/{repo}/pulls/{pull_number}/files
  → Returns list of modified file paths

Step 4: Match files → owners
  → For each changed file, find matching CODEOWNERS patterns
  → Collect the set of required code owners (users + teams)
  → If a team is listed (e.g., @org/team-name), resolve team members via:
     GET /orgs/{org}/teams/{team_slug}/members

Step 5: Check reviews for code owner approval
  → GET /repos/{owner}/{repo}/pulls/{pull_number}/reviews
  → If any reviewer in the code owners set has state = "APPROVED" → ✅
  → Otherwise → ⏳
```

> **Note:** There is no single GitHub API that directly tells you "code owner has approved." We must compose this from the CODEOWNERS file + changed files + reviews. The CODEOWNERS file is fetched once and cached (it changes infrequently). The file parsing follows the same glob-matching rules GitHub uses internally.

---

### FR-11: PR Detail Panel (Side Drawer)

| Attribute | Detail |
|-----------|--------|
| **ID** | FR-11 |
| **Title** | Show Detailed PR View in Side Panel |
| **Description** | When a user clicks on a PR row in the table, a **side panel (drawer)** shall slide in from the right, displaying detailed information about that PR without navigating away from the dashboard. |
| **Acceptance Criteria** | 1. Clicking a PR row opens a right-side drawer panel. <br> 2. The main table remains visible (narrowed) for context. <br> 3. The drawer can be dismissed by clicking outside, pressing Escape, or a close button. <br> 4. The drawer displays the detailed information listed below. |

#### Detail Panel Contents

| # | Section | Data Shown |
|---|---------|------------|
| 1 | **Header** | PR title, number, author avatar + username, branch badge (Main/Feature) |
| 2 | **Status Bar** | Age (with color indicator), code owner approval status (✅/⏳) |
| 3 | **Description** | PR body/description (rendered Markdown, truncated with "show more") |
| 4 | **Branches** | Source branch → Base branch |
| 5 | **Reviewers** | List of active reviewers with their review status (approved ✅ / changes requested 🔄 / commented 💬) |
| 6 | **Code Owners** | List of identified code owners for this PR and whether they've approved |
| 7 | **Files Changed** | Count of files changed (additions/deletions summary) |
| 8 | **Labels** | Any labels applied to the PR |
| 9 | **Actions** | "Open in GitHub" button (new tab) |

#### UX Design Rationale

| Pattern | Pros | Cons | **Decision** |
|---------|------|------|--------------|
| Expandable Row (accordion) | Simple, inline | Limited space, pushes other rows down | ❌ |
| Side Panel (drawer) | Table stays visible, ample detail space, standard pattern | Slightly more complex to implement | ✅ **Selected** |
| Modal / Dialog | Focused view | Blocks table, feels disruptive | ❌ |
| Separate Page | Maximum space | Loses table context, extra navigation | ❌ |

> **Recommendation:** The **right-side drawer** is the best pattern for this use case. It allows users to quickly scan PR details while keeping the full table visible for context. This is a standard dashboard pattern (used by Jira, Linear, GitHub Projects, etc.).

## 5. Dashboard View - Data Columns (Phase 1)

The dashboard uses a **single table with tab filters** (All | Main | Feature Branches).

| # | Column | Description |
|---|--------|-------------|
| 1 | **PR Number** | The pull request number (e.g., #142) |
| 2 | **Title** | The title of the pull request |
| 3 | **Author** | GitHub username of the PR creator |
| 4 | **Base Branch** | Target branch with type badge (Main / Feature) |
| 5 | **Age** | Time since PR was opened (e.g., "3d 4h"), highlighted if stale |
| 6 | **Active Reviewers** | Count of reviewers who have actively engaged |
| 7 | **Code Owner Status** | ✅ Approved / ⏳ Pending (shown for Main PRs) |
| 8 | **Link** | Direct URL to the PR on GitHub |

**Filter Tabs:** All | Main | Feature Branches  
**Row Click:** Opens the PR Detail Panel (side drawer)

## 6. User Stories

### US-1: View Open PRs
> **As a** team member,  
> **I want to** see all open PRs from my team in one dashboard,  
> **So that** I can quickly identify which PRs need attention.

### US-2: Identify Stale PRs
> **As a** team lead,  
> **I want to** see how long each PR has been open,  
> **So that** I can follow up on PRs that are aging and may be blocked.

### US-3: Check Active Reviewer Engagement
> **As a** PR author,  
> **I want to** see how many reviewers have **actively** reviewed my PR (commented/approved/requested changes),  
> **So that** I know if my PR is actually being reviewed or if I need to follow up.

### US-4: Quick Navigation
> **As a** team member,  
> **I want to** click on a PR and be taken directly to GitHub,  
> **So that** I can review or take action on it immediately.

### US-5: Filter by Branch Type
> **As a** team member,  
> **I want to** filter PRs by whether they target `main` or a `feature/` branch,  
> **So that** I can focus on the PRs relevant to my current review workflow.

### US-6: Automatic Team Resolution
> **As a** team lead,  
> **I want** the dashboard to automatically fetch team members from our GitHub team (PHOENIX),  
> **So that** I don't have to manually maintain a list of usernames.

### US-7: Check Code Owner Approval
> **As a** PR author,  
> **I want to** see whether a code owner has approved my PR,  
> **So that** I know if the PR is ready to merge from a code ownership perspective.

### US-8: View PR Details
> **As a** team member,  
> **I want to** click on a PR row and see detailed information in a side panel,  
> **So that** I can get full context (description, reviewers, files changed) without leaving the dashboard.

## 7. Phase 2 Enhancements (Out of Scope for Phase 1)

The following are explicitly **not** in scope for Phase 1 but planned for future phases:

| # | Enhancement | Description |
|---|-------------|-------------|
| 1 | **Team Management UI** | UI to search GitHub org members and add/remove them from the tracked team |
| 2 | Detailed reviewer names/status in table | Show reviewer names directly in the table (currently only in detail panel) |
| 3 | PR review comments count | Number of review comments per PR |
| 4 | CI/CD status per PR | Build/test status indicators |
| 5 | Notifications/alerts | Slack, email alerts for stale PRs |
| 6 | Multi-repository support | Track PRs across multiple repos |
| 7 | Historical trends / analytics | PR velocity, merge time trends |
| 8 | PR merge time tracking | Time from open to merge |

## 8. Assumptions

1. A GitHub team named **PHOENIX** (slug: `phoenix`) exists in the organization and contains all relevant team members.
2. A valid GitHub token with `repo` + `read:org` scopes will be provided.
3. The target repository is hosted on GitHub.com (not GitHub Enterprise, unless token supports it).
4. The dashboard will be accessed by team members within a browser.
5. PRs targeting branches starting with `feature/` are considered feature branch PRs; all others are considered main PRs.
6. Code owners are auto-assigned as reviewers but should NOT count toward active review engagement unless they take action.
7. A `CODEOWNERS` file exists in the repository (in `.github/`, root, or `docs/` directory).
8. The CODEOWNERS file follows standard GitHub CODEOWNERS syntax (glob patterns).

## 9. Open Questions

| # | Question | Status |
|---|----------|--------|
| 1 | Which GitHub organization and repository (org/repo) should be tracked? | **Pending - input from user** |
| 2 | Confirm team slug is `phoenix`? | **Pending - input from user** |
| 3 | Should the aging threshold for highlighting be configurable? Default: 3 days | Pending |
| 4 | Is this GitHub.com or GitHub Enterprise? | Pending |
| 5 | Any authentication preferences (PAT vs GitHub App)? | Pending |
| 6 | Are there base branches other than `main` and `feature/*` that need special handling? | Pending |

---

## Next Steps

1. **User to confirm:** Organization, repository, and team slug.
2. **Next document:** Technical Requirements Document (architecture, tech stack, API design).
3. **Then:** Implementation.

# GitHub PR Tracker - Functional Requirements Document

**Project:** GitHub PR Tracker Dashboard  
**Version:** 1.0 (Phase 1)  
**Date:** 2 June 2026  
**Status:** Draft (Rev 2)

## 1. Overview

The GitHub PR Tracker is a dashboard application that provides visibility into all **open Pull Requests** for a given GitHub repository, filtered to PRs opened by members of a configured GitHub team (PHOENIX). It enables the team to monitor PR status, aging, active reviewer engagement, and branch targeting at a glance.

## 2. Goals & Objectives

| # | Objective |
|---|---|
| 1 | Provide a single view of all open PRs for the team across a repository |
| 2 | Highlight how long each PR has been open (aging) |
| 3 | Show **active** reviewer count per PR (not just assigned/code-owner reviewers) |
| 4 | Identify the PR author |
| 5 | Enable quick filtering and sorting for prioritization |
| 6 | Distinguish between PRs targeting `main` vs `feature/` branches |

## 3. Inputs / Configuration

| Input | Description | Example |
|-------|-------------|--------|
| **Organization** | The GitHub organization that owns the repo and team | `my-org` |
| **Repository** | The GitHub repository to track (owner/repo) | `my-org/my-repo` |
| **Team Slug** | The GitHub team slug to fetch members from | `phoenix` |
| **GitHub Token** | A PAT or GitHub App token with `repo` + `read:org` scopes | — |
| **Aging Threshold** | Number of days after which a PR is considered stale (default: 3) | `3` |

## 4. Functional Requirements

### FR-1: Fetch Open PRs

| Attribute | Detail |
|-----------|--------|
| **ID** | FR-1 |
| **Title** | Fetch Open Pull Requests |
| **Description** | The system shall fetch all currently open pull requests from the configured GitHub repository. |
| **Acceptance Criteria** | 1. All open PRs are retrieved via the GitHub API. <br> 2. Pagination is handled for repositories with large numbers of PRs. |

---

### FR-2: Resolve Team Members from GitHub Team

| Attribute | Detail |
|-----------|--------|
| **ID** | FR-2 |
| **Title** | Fetch Team Members from GitHub Organization Team (PHOENIX) |
| **Description** | The system shall resolve the list of team members by fetching members of the configured GitHub team (e.g., `PHOENIX`) via the GitHub API (`GET /orgs/{org}/teams/{team_slug}/members`). The fetched PRs are then filtered to only display those opened by these team members. |
| **Acceptance Criteria** | 1. Team members are fetched dynamically from the GitHub team using the API. <br> 2. Only PRs authored by resolved team members appear in the dashboard. <br> 3. If the API call fails (e.g., insufficient permissions), the system falls back to a static/configured list of usernames. <br> 4. The GitHub token must have `read:org` scope for this to work. |
| **Phase 2 Enhancement** | A UI will allow users to search GitHub org members and manually add/remove members from the tracked team list. |

> **Technical Note:** The GitHub API endpoint `GET /orgs/{org}/teams/{team_slug}/members` returns all members of a team. The token requires `read:org` scope. This eliminates the need to manually maintain a user list in Phase 1.

---

### FR-3: Display PR Author

| Attribute | Detail |
|-----------|--------|
| **ID** | FR-3 |
| **Title** | Show PR Author |
| **Description** | For each open PR, the system shall display the GitHub username (and optionally avatar) of the author who opened the PR. |
| **Acceptance Criteria** | 1. Author username is visible for every PR row. <br> 2. Author is correctly mapped to the team member list. |

---

### FR-4: Display PR Age (Time Open)

| Attribute | Detail |
|-----------|--------|
| **ID** | FR-4 |
| **Title** | Show How Long Each PR Has Been Open |
| **Description** | The system shall calculate and display the duration (in days/hours) since each PR was opened. |
| **Acceptance Criteria** | 1. Age is displayed in a human-readable format (e.g., "3 days", "5 hours"). <br> 2. PRs open longer than a configurable threshold (default: 3 days) are visually highlighted (e.g., red/amber). |

---

### FR-5: Display Active Reviewer Count

| Attribute | Detail |
|-----------|--------|
| **ID** | FR-5 |
| **Title** | Show Number of **Active** Reviewers |
| **Description** | For each PR, the system shall display the count of reviewers who have **actively engaged** with the PR. Active engagement is defined as: submitting a review (approve / request changes), or leaving a comment on the PR. Auto-assigned reviewers (e.g., via CODEOWNERS) who have not taken any action are **excluded** from this count. |
| **Acceptance Criteria** | 1. Only reviewers who have commented, approved, or requested changes are counted. <br> 2. Reviewers who are merely assigned/requested but have not acted are NOT counted. <br> 3. Count is displayed as a numeric value per PR. <br> 4. Data source: PR reviews API (`GET /repos/{owner}/{repo}/pulls/{pull_number}/reviews`) and PR comments. |
| **Rationale** | Code owners are auto-assigned to every PR but do not always actively review. Showing assigned count would be misleading; active count reflects actual review engagement. |

---

### FR-6: Display PR Metadata

| Attribute | Detail |
|-----------|--------|
| **ID** | FR-6 |
| **Title** | Show PR Title, Number, and Link |
| **Description** | Each PR in the dashboard shall display its title, PR number, and a clickable link to the PR on GitHub. |
| **Acceptance Criteria** | 1. PR title is shown. <br> 2. PR number (e.g., #142) is shown. <br> 3. Clicking the PR opens it in a new browser tab on GitHub. |

---

### FR-7: Sort PRs

| Attribute | Detail |
|-----------|--------|
| **ID** | FR-7 |
| **Title** | Sort PRs by Age, Author, or Active Reviewer Count |
| **Description** | The dashboard shall allow sorting of the PR list by age (oldest first), author name, or active reviewer count. |
| **Acceptance Criteria** | 1. Default sort is by age (oldest first). <br> 2. User can toggle between sort criteria. |

---

### FR-8: Auto-Refresh / Manual Refresh

| Attribute | Detail |
|-----------|--------|
| **ID** | FR-8 |
| **Title** | Refresh PR Data |
| **Description** | The dashboard shall support refreshing the data either on a configurable interval (auto-refresh) or via a manual refresh button. |
| **Acceptance Criteria** | 1. A refresh button is available. <br> 2. Optionally, data refreshes automatically every N minutes (configurable). |

---

### FR-9: Filter by Base Branch (Main vs Feature)

| Attribute | Detail |
|-----------|--------|
| **ID** | FR-9 |
| **Title** | Filter PRs by Target Base Branch |
| **Description** | The dashboard shall categorize and allow filtering of PRs based on their target (base) branch. PRs are classified as: <br> - **Main PRs** — base branch is `main` (or `master`) <br> - **Feature PRs** — base branch starts with `feature/` |
| **Acceptance Criteria** | 1. Each PR displays a branch type indicator (badge/chip: "Main" or "Feature"). <br> 2. The dashboard provides tab/toggle filters: **All** \| **Main** \| **Feature Branches**. <br> 3. Classification rule: if base branch starts with `feature/`, it is a Feature PR; otherwise it is a Main PR. <br> 4. The selected filter persists during the session. |
| **Rationale** | Feature branch PRs do not require code owners review and have different review dynamics. Separating them helps the team focus on the PRs relevant to their workflow. |
| **UX Recommendation** | Use a **single table with tab filters** (All / Main / Feature) rather than two separate tables. This keeps the UI clean, avoids layout duplication, and lets users see everything in one place or drill down as needed. |

## 5. Dashboard View - Data Columns (Phase 1)

The dashboard uses a **single table with tab filters** (All | Main | Feature Branches).

| # | Column | Description |
|---|--------|-------------|
| 1 | **PR Number** | The pull request number (e.g., #142) |
| 2 | **Title** | The title of the pull request |
| 3 | **Author** | GitHub username of the PR creator |
| 4 | **Base Branch** | Target branch with type badge (Main / Feature) |
| 5 | **Age** | Time since PR was opened (e.g., "3d 4h"), highlighted if stale |
| 6 | **Active Reviewers** | Count of reviewers who have actively engaged (commented/approved/requested changes) |
| 7 | **Link** | Direct URL to the PR on GitHub |

**Filter Tabs:** All | Main | Feature Branches

## 6. User Stories

### US-1: View Open PRs
> **As a** team member,  
> **I want to** see all open PRs from my team in one dashboard,  
> **So that** I can quickly identify which PRs need attention.

### US-2: Identify Stale PRs
> **As a** team lead,  
> **I want to** see how long each PR has been open,  
> **So that** I can follow up on PRs that are aging and may be blocked.

### US-3: Check Active Reviewer Engagement
> **As a** PR author,  
> **I want to** see how many reviewers have **actively** reviewed my PR (commented/approved/requested changes),  
> **So that** I know if my PR is actually being reviewed or if I need to follow up.

### US-4: Quick Navigation
> **As a** team member,  
> **I want to** click on a PR and be taken directly to GitHub,  
> **So that** I can review or take action on it immediately.

### US-5: Filter by Branch Type
> **As a** team member,  
> **I want to** filter PRs by whether they target `main` or a `feature/` branch,  
> **So that** I can focus on the PRs relevant to my current review workflow (e.g., main PRs that need code owner reviews).

### US-6: Automatic Team Resolution
> **As a** team lead,  
> **I want** the dashboard to automatically fetch team members from our GitHub team (PHOENIX),  
> **So that** I don't have to manually maintain a list of usernames.

## 7. Phase 2 Enhancements (Out of Scope for Phase 1)

The following are explicitly **not** in scope for Phase 1 but planned for future phases:

| # | Enhancement | Description |
|---|-------------|-------------|
| 1 | **Team Management UI** | UI to search GitHub org members and add/remove them from the tracked team (instead of relying solely on the GitHub team API) |
| 2 | Detailed reviewer names/status | Show who approved, requested changes, or is pending |
| 3 | PR review comments count | Number of review comments per PR |
| 4 | CI/CD status per PR | Build/test status indicators |
| 5 | Notifications/alerts | Slack, email alerts for stale PRs |
| 6 | Multi-repository support | Track PRs across multiple repos |
| 7 | Historical trends / analytics | PR velocity, merge time trends |
| 8 | PR merge time tracking | Time from open to merge |

## 8. Assumptions

1. A GitHub team named **PHOENIX** (slug: `phoenix`) exists in the organization and contains all relevant team members.
2. A valid GitHub token with `repo` + `read:org` scopes will be provided.
3. The target repository is hosted on GitHub.com (not GitHub Enterprise, unless token supports it).
4. The dashboard will be accessed by team members within a browser.
5. PRs targeting branches starting with `feature/` are considered feature branch PRs; all others are considered main PRs.
6. Code owners are auto-assigned as reviewers but should NOT count toward active review engagement unless they take action (comment/approve/request changes).

## 9. Open Questions

| # | Question | Status |
|---|----------|--------|
| 1 | Which GitHub organization and repository (org/repo) should be tracked? | **Pending - input from user** |
| 2 | Confirm team slug is `phoenix`? | **Pending - input from user** |
| 3 | Should the aging threshold for highlighting be configurable? Default: 3 days | Pending |
| 4 | Is this GitHub.com or GitHub Enterprise? | Pending |
| 5 | Any authentication preferences (PAT vs GitHub App)? | Pending |
| 6 | Are there base branches other than `main` and `feature/*` that need special handling? | Pending |

---

## Next Steps

1. **User to confirm:** Organization, repository, and team slug.
2. **Next document:** Technical Requirements Document (architecture, tech stack, API design).
3. **Then:** Implementation.

## 14. Phase 4 Requirements — UX/UI Refinement & Team/Person Filters

Phase 4 focuses on reducing dashboard clutter, improving information hierarchy, and introducing better filtering UX for team leads and reviewers.

### 14.1 Feasibility Assessment (Before Requirements)

| Requested Enhancement | Feasibility | Notes / Constraints |
|---|---|---|
| Show **display names** instead of GitHub usernames | **Feasible with minor backend work** | Team-members endpoint usually returns `login`; to get `name`, backend must hydrate users from `/users/{login}` or GraphQL `User.name`. Need fallback to `login` when name is null. |
| Support **4 team slugs** (`phoenix`, `nexus`, `macgyver`, `bespin`) | **Feasible with moderate backend + filter changes** | Replace single `GITHUB_TEAM_SLUG` with list config (e.g., `GITHUB_TEAM_SLUGS`). Need de-dup users across teams. UI can provide team-switch / multi-select. |
| Significant UX/UI cleanup + better dropdowns | **Fully feasible** | Frontend-only redesign mostly. Can keep same data model while improving layout, spacing, and visual priority. |
| Show PR count at bottom like pagination summary | **Feasible and recommended** | Since dataset is small, full server pagination is optional. UX can show `Showing X–Y of Z` and optional client-side page size selector. |
| Add per-person filter | **Fully feasible** | Client-side filter by author is straightforward once author display names are available. |

> **Pagination note:** Given the small dataset, full backend pagination is not required now. Phase 4 should implement **pagination-style footer metadata** and optional **client-side pagination**. Backend pagination can be deferred unless row counts grow.

---

### FR-22: Display Names Instead of Usernames

| Attribute | Detail |
|---|---|
| **ID** | FR-22 |
| **Title** | Show Human-Friendly User Display Names |
| **Description** | The dashboard shall display user **name** (e.g., "Ravi Nimje") instead of GitHub login (e.g., `rnimje-nasuni`) wherever possible, including PR author, reviewers, and filters. |
| **Acceptance Criteria** | 1. Author column shows `display_name` when available. <br> 2. Tooltip / detail panel sections (team approvals, reviewers) use display names. <br> 3. If name is missing/null, fallback to username/login. <br> 4. API response includes both fields for compatibility: `display_name`, `username`. |
| **Data Contract** | Add `display_name` to Author and Reviewer models without removing existing `username`. |

---

### FR-23: Multi-Team Support (4 Team Slugs)

| Attribute | Detail |
|---|---|
| **ID** | FR-23 |
| **Title** | Support Multiple Team Slugs |
| **Description** | The system shall support PR tracking for four teams: `phoenix`, `nexus`, `macgyver`, `bespin`, and allow viewing PRs by selected team(s). |
| **Acceptance Criteria** | 1. Backend accepts multiple team slugs via config (e.g., `GITHUB_TEAM_SLUGS=phoenix,nexus,macgyver,bespin`). <br> 2. Team member resolution merges users across selected teams (de-duplicated). <br> 3. UI shows team selector (single-select by default, optional multi-select). <br> 4. Default selected team is `phoenix` for backward compatibility. <br> 5. Selected team filter affects PR list, per-person filter options, and counts. |
| **Out of Scope** | Team permission management and role-based access control. |

---

### FR-24: UX/UI Refinement for Clarity

| Attribute | Detail |
|---|---|
| **ID** | FR-24 |
| **Title** | Redesign Dashboard for Better Visual Hierarchy |
| **Description** | The dashboard shall be redesigned to reduce clutter and improve scanability, prioritizing team, repository, author, age risk, and actionability. |
| **Acceptance Criteria** | 1. Replace current native/basic dropdowns with accessible, styled combobox/select components. <br> 2. Move filters into a clear control bar: Team selector, Repository selector, Person selector, Branch tabs, Search input (optional). <br> 3. Improve table readability: tighter columns, consistent badge sizes, aligned numeric columns, sticky header. <br> 4. Maintain quick actions (open PR, detail drawer) with less visual noise. <br> 5. Ensure responsive behavior on laptop widths without truncation issues. |
| **UX Direction** | Emphasize actionable states: stale age, unresolved comments, approvals readiness. De-emphasize low-signal metadata. |

#### Recommended Table Information Priority (Phase 4)

| Priority | Column | Keep / Change |
|---|---|---|
| High | PR # + Title | Keep (primary cell with unresolved indicator) |
| High | Author (display name) | Keep (switch to display name) |
| High | Team | Add (for multi-team context) |
| High | Repo | Keep |
| High | Age | Keep (risk color emphasis) |
| Medium | Team Approvals | Keep |
| Medium | Reviewers | Keep (compact) |
| Medium | Code Owner | Keep (compact badge) |
| Low | External Link | Keep as icon-only action |

---

### FR-25: Pagination-Style Footer and PR Count

| Attribute | Detail |
|---|---|
| **ID** | FR-25 |
| **Title** | Show Bottom Summary and Pagination Controls |
| **Description** | The table shall display a footer with PR counts and pagination-style navigation for improved orientation, even with smaller datasets. |
| **Acceptance Criteria** | 1. Footer shows `Showing X–Y of Z PRs`. <br> 2. Footer updates with active filters (team/repo/person/branch). <br> 3. Add page size selector (10, 25, 50, All). <br> 4. Add Prev/Next controls for client-side pagination. <br> 5. If total rows ≤ selected page size, controls remain visible but disabled appropriately. |
| **Implementation Note** | Client-side pagination in Phase 4; backend pagination deferred unless required by dataset growth. |

---

### FR-26: Per-Person Filter

| Attribute | Detail |
|---|---|
| **ID** | FR-26 |
| **Title** | Filter PRs by Author (Person) |
| **Description** | Users shall be able to filter PRs by individual author from a person dropdown/autocomplete. |
| **Acceptance Criteria** | 1. Person filter supports `All People` default. <br> 2. Person options are derived from current dataset (respecting selected team). <br> 3. Person labels use display names; fallback to username. <br> 4. Person filter composes with team/repo/branch filters. <br> 5. Counts and footer summary reflect person filter instantly (client-side). |

---

## 15. Phase 4 User Stories

### US-17: Readable Names
> As a reviewer, I want to see real names instead of GitHub usernames, so I can identify teammates faster.

### US-18: Team Scope Switching
> As an engineering manager, I want to switch between Phoenix, Nexus, Macgyver, and Bespin views, so I can monitor each team separately.

### US-19: Cleaner Controls
> As a daily dashboard user, I want modern dropdowns and cleaner filter placement, so I can use the tracker quickly without visual clutter.

### US-20: Bottom Count Clarity
> As a user, I want a bottom summary like `Showing 1–10 of 34`, so I understand list size and my current view.

### US-21: Person Drill-Down
> As a lead, I want to filter by person, so I can review individual PR load and bottlenecks.

---

## 16. Open Questions for Phase 4 Review

| # | Question | Suggested Default |
|---|---|---|
| 1 | Team selector behavior: single-select or multi-select? | Start single-select; add multi-select later if needed |
| 2 | Should person filter include reviewers too, or only PR authors? | Start with authors only |
| 3 | Should pagination default be 10 or 25 rows? | 10 for readability |
| 4 | Should display names be cached server-side? | Yes (TTL cache) |
| 5 | Should table support free-text search in Phase 4? | Optional but recommended |

## 17. Additional UX Requirements (Recommended for Phase 4)

These are optional-but-high-value UX improvements that can significantly improve day-to-day usability and reduce cognitive load.

### FR-27: Dashboard Summary Bar (Top KPIs)

| Attribute | Detail |
|---|---|
| **ID** | FR-27 |
| **Title** | Add KPI Summary Cards Above Table |
| **Description** | Show quick summary metrics so users can understand workload without scanning rows. |
| **Acceptance Criteria** | 1. Display cards for: Total Open PRs, Stale PRs, PRs with unresolved comments, PRs ready for code-owner ping. <br> 2. Metrics respect active filters (team/repo/person/branch). <br> 3. Clicking a KPI applies a corresponding quick filter (optional in first iteration). |

---

### FR-28: Strong Loading / Empty / Error States

| Attribute | Detail |
|---|---|
| **ID** | FR-28 |
| **Title** | Improve Non-Happy Path UX |
| **Description** | The dashboard shall provide polished states for loading, no-results, and API failures. |
| **Acceptance Criteria** | 1. Use skeleton rows while loading table data. <br> 2. Empty state message should explain why no rows are shown (e.g., filters too restrictive). <br> 3. Error state includes clear retry action and last successful refresh time if available. <br> 4. No sudden layout jumps between states. |

---

### FR-29: Advanced Filter UX (Chips + Clear All)

| Attribute | Detail |
|---|---|
| **ID** | FR-29 |
| **Title** | Make Active Filters Obvious and Easy to Reset |
| **Description** | Users shall see active filters as chips and have one-click reset controls. |
| **Acceptance Criteria** | 1. Active filters appear as chips below control bar. <br> 2. Each chip can be removed individually. <br> 3. A `Clear all` action resets to defaults. <br> 4. Filter state persists during session refresh/navigation. |

---

### FR-30: User-Controlled Table Density

| Attribute | Detail |
|---|---|
| **ID** | FR-30 |
| **Title** | Add Compact / Comfortable Row Density Modes |
| **Description** | Users shall be able to switch between compact and comfortable table density. |
| **Acceptance Criteria** | 1. Density toggle available near controls. <br> 2. Compact mode increases visible rows without breaking readability. <br> 3. Preference persists in local storage. |

---

### FR-31: Column Visibility & Ordering Preferences

| Attribute | Detail |
|---|---|
| **ID** | FR-31 |
| **Title** | Allow Users to Customize Visible Columns |
| **Description** | Users shall control which columns are visible to match personal workflow. |
| **Acceptance Criteria** | 1. Column chooser popover with checkboxes. <br> 2. Essential columns (`PR #`, `Title`, `Author`) cannot be hidden. <br> 3. Column preferences persist per user/browser. |

---

### FR-33: Save and Reuse Filter Presets

| Attribute | Detail |
|---|---|
| **ID** | FR-33 |
| **Title** | Save Filter Presets for Frequent Workflows |
| **Description** | Users shall save named filter views (e.g., `My Team + Stale`, `Nexus + Main`). |
| **Acceptance Criteria** | 1. Users can save current filter state with a name. <br> 2. Presets can be selected, renamed, and deleted. <br> 3. Default preset auto-loads when dashboard opens. |
| **Implementation Note** | Start with browser-local storage; server-side shared presets can be a later phase. |

---

## 18. Prioritization Recommendation

| Priority | Items |
|---|---|
| **Must Have (Phase 4 core)** | FR-22, FR-23, FR-24, FR-25, FR-26 |
| **Should Have (high UX impact)** | FR-27, FR-28, FR-29 |
| **Could Have (power-user enhancements)** | FR-30, FR-31, FR-33 |

### FR-35: Correct Merge Eligibility Logic for KPI

| Attribute | Detail |
|---|---|
| **ID** | FR-35 |
| **Title** | Redefine "Ready to Merge" Based on Branch-Specific Policy |
| **Description** | The merge-readiness KPI shall use branch-aware policy rules instead of generic approvals count. |
| **Acceptance Criteria** | 1. **For PRs targeting `main`:** Ready-to-merge = at least 1 code-owner approval **AND** at least 1 additional team member approval. Minimum total: 2 approvals with 1 code-owner + 1 team member. <br> 2. **For PRs targeting `feature/*`:** Ready-to-merge = at least 2 team member approvals (code-owner approval optional). <br> 3. KPI card label and tooltip explain branch-specific rule. <br> 4. PR row status (if shown) uses the same logic as KPI to avoid inconsistency. |
| **Supersedes** | Clarifies and replaces any ambiguous wording in FR-27 KPI semantics. |